# AI Tools & Mini Project
## Part 1: Explore AI tools | Part 2: Use AI to code, research & be productive | Part 3: Build an AI-powered app

**Mini project:** we upgrade the earlier **Expense Tracker** into an **AI Expense Assistant**. You type expenses in plain English ("lunch with friends was 12.50 yesterday") and an AI model turns them into clean, structured data, then gives you spending insights.

> You can run the whole notebook **without any API key**. It falls back to a simple rule-based parser, and you'll see exactly where AI does better.

---
# Part 1: Exploring AI Tools

| Tool | Made by | Where you find it | Good for |
|---|---|---|---|
| **ChatGPT** | OpenAI | chatgpt.com, mobile apps | General chat, writing, brainstorming, coding help, explaining concepts |
| **Gemini** | Google | gemini.google.com, Google Workspace, Colab | Google-integrated work (Docs, Gmail, Drive), long documents, free API tier for students |
| **Microsoft Copilot** | Microsoft | copilot.microsoft.com, Windows, Edge, Word/Excel/PowerPoint | Office-document tasks, web-connected answers, GitHub Copilot for in-editor code completion |
| **Claude** | Anthropic | claude.ai | Long-form writing, careful analysis, coding, working with documents |

> Features, free limits, and model names change often. Check each tool's website for the latest.

### Try this (10 minutes)
Give the **same prompt** to two or three tools and compare the answers:

```
Explain Python dictionaries to a beginner in 5 sentences, then give me
a 3-question quiz with answers.
```

| What to compare | Tool A | Tool B | Tool C |
|---|---|---|---|
| Was it accurate? | | | |
| Was it clear for a beginner? | | | |
| Did it follow the format I asked for? | | | |
| Speed / ease of use | | | |

---
# Part 2: How AI Tools Help

## Coding
| Use | Example prompt |
|---|---|
| **Explain code** | *"Explain this code line by line like I'm a beginner: [paste code]"* |
| **Debug** | *"I get `KeyError: 'amount'` in this code. What does it mean and how do I fix it? [paste code + error]"* |
| **Generate** | *"Write a Python function that validates an email address, with 5 test cases."* |
| **Improve** | *"Refactor this function to be shorter and add comments."* |
| **Learn** | *"Give me 5 practice exercises on list comprehensions, easiest to hardest."* |

## Research
| Use | Example prompt |
|---|---|
| **Summarize** | *"Summarize this article in 5 bullet points: [paste text]"* |
| **Understand** | *"Explain overfitting in machine learning using a school exam analogy."* |
| **Compare** | *"Compare logistic regression and decision trees in a table."* |
| **Plan** | *"Make a 4-week study plan to learn Pandas, 30 minutes a day."* |

## Productivity
| Use | Example prompt |
|---|---|
| **Write** | *"Draft a polite email asking my teacher for an extension."* |
| **Organize** | *"Turn these messy notes into a to-do list sorted by priority."* |
| **Brainstorm** | *"Give me 10 mini-project ideas using Python and CSV files."* |

## A good prompt formula
**Role + Task + Context + Format**

| | Weak prompt | Strong prompt |
|---|---|---|
| Example | *"Fix my code"* | *"You are a Python tutor. My function should return the average of a list but returns 0 for empty lists. Here is the code and error: [...]. Explain the bug, then show the fixed code."* |

## Use AI responsibly
- **Verify everything.** AI can sound confident and still be wrong ("hallucination"). Run the code, check facts and citations.
- **Don't paste secrets.** No passwords, API keys, or private personal data.
- **Understand before you copy.** If you can't explain the code, you haven't learned it.
- **Follow your school or workplace rules** on AI use, and say when AI helped.

---
# Part 3: Mini Project: AI Expense Assistant

**Plan**
1. Set up (API key optional)
2. Write a function that talks to the AI
3. Turn plain-English sentences into structured expenses
4. **Validate** the AI's output (never trust it blindly)
5. Analyze with Pandas and chart with Matplotlib
6. Ask the AI for spending advice

### Step 1: Setup

In [ ]:
!pip install -q google-genai

**Optional: enable the AI.** Get a free key at [aistudio.google.com/apikey](https://aistudio.google.com/apikey), then in Colab click the **key icon** (Secrets) in the left sidebar, add a secret named `GEMINI_API_KEY`, and switch **Notebook access** on. Never paste a key directly into notebook cells you plan to share.

In [ ]:
import os, json, re
from datetime import date, timedelta

import pandas as pd
import matplotlib.pyplot as plt

# --- Look for an API key (Colab secret first, then environment variable) ---
API_KEY = None
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

USE_AI = bool(API_KEY)
MODEL = "gemini-2.5-flash"   # if this name is retired, check Google AI Studio for a current one

if USE_AI:
    from google import genai
    client = genai.Client(api_key=API_KEY)
    print(f"AI mode ON, using model: {MODEL}")
else:
    print("No API key found: running in OFFLINE mode (rule-based parser).")
    print("Add GEMINI_API_KEY to Colab Secrets to switch the AI on.")

### Step 2: A function that talks to the AI

In [ ]:
def ask_ai(prompt, json_mode=False):
    """Send a prompt to Gemini and return the text reply."""
    config = {"response_mime_type": "application/json"} if json_mode else None
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text

In [ ]:
# Quick test (only runs if AI is on)
if USE_AI:
    print(ask_ai("In one sentence, what is a Python dictionary?"))
else:
    print("Skipped: AI is off.")

### Step 3: Turn plain English into structured data
The key idea: **we tell the AI exactly what format we want** (JSON with fixed fields and allowed categories), so our program can use the answer.

First, shared settings and a **validator**. AI output can be wrong or messy, so we check every field before trusting it.

In [ ]:
CATEGORIES = ["Food", "Travel", "Bills", "Shopping", "Entertainment", "Health", "Other"]

def validate(record):
    """Clean up and check one expense record. Raises ValueError if unusable."""
    amount = float(record["amount"])
    if amount <= 0:
        raise ValueError("amount must be positive")

    category = str(record.get("category", "Other")).title()
    if category not in CATEGORIES:
        category = "Other"

    try:
        when = date.fromisoformat(str(record.get("date")))
    except ValueError:
        when = date.today()

    return {
        "date": when,
        "description": str(record.get("description", "Unnamed")).strip() or "Unnamed",
        "category": category,
        "amount": round(amount, 2),
    }

**Option A: the AI parser.** The prompt gives the AI a role, the task, today's date (so it can work out "yesterday"), the allowed categories, and the exact output format.

In [ ]:
def parse_expense_ai(text):
    prompt = f"""You are an expense-tracking assistant.
Extract ONE expense from the user's sentence.

Today's date is {date.today().isoformat()}.
Allowed categories: {", ".join(CATEGORIES)}.

Return ONLY a JSON object with these keys:
  "description": short text (max 5 words)
  "category": one of the allowed categories
  "amount": the TOTAL amount paid, as a number
  "date": YYYY-MM-DD (use today's date if none is mentioned)

User sentence: "{text}"
"""
    reply = ask_ai(prompt, json_mode=True)
    reply = re.sub(r"```json|```", "", reply).strip()   # strip code fences if any
    return validate(json.loads(reply))

**Option B: a rule-based fallback.** It uses simple regex and keyword matching. It's fast and free, but fragile. Watch what happens with tricky sentences.

In [ ]:
KEYWORDS = {
    "Food": ["lunch", "dinner", "breakfast", "coffee", "groceries", "pizza", "restaurant", "snack"],
    "Travel": ["bus", "taxi", "uber", "train", "fuel", "petrol", "flight", "ticket"],
    "Bills": ["bill", "electricity", "rent", "internet", "water", "phone"],
    "Shopping": ["bought", "shirt", "shoes", "clothes", "shopping"],
    "Entertainment": ["movie", "game", "concert", "netflix"],
    "Health": ["pharmacy", "doctor", "medicine", "gym"],
}
NUMBER_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7}

def parse_expense_rules(text):
    lower = text.lower()

    # Amount: first number in the sentence
    match = re.search(r"\d+(?:\.\d+)?", lower)
    if not match:
        raise ValueError("no amount found")
    amount = float(match.group())

    # Category: first keyword match
    category = "Other"
    for cat, words in KEYWORDS.items():
        if any(w in lower for w in words):
            category = cat
            break

    # Date: "yesterday" or "N days ago"
    when = date.today()
    if "yesterday" in lower:
        when -= timedelta(days=1)
    else:
        ago = re.search(r"(\d+|" + "|".join(NUMBER_WORDS) + r")\s+days?\s+ago", lower)
        if ago:
            n = ago.group(1)
            when -= timedelta(days=int(n) if n.isdigit() else NUMBER_WORDS[n])

    return validate({"description": text[:30], "category": category,
                     "amount": amount, "date": when.isoformat()})

**Combine them:** use the AI when it's available, and fall back to the rules if the AI call fails (no internet, rate limit, bad output).

In [ ]:
def parse_expense(text):
    if USE_AI:
        try:
            return parse_expense_ai(text)
        except Exception as err:
            print(f"  (AI failed: {err}. Using rules instead.)")
    return parse_expense_rules(text)

### Step 4: Try it on realistic sentences
Notice the last two: *"2 coffees for 9 total"* and *"two days ago"* are traps for the rule-based parser (it grabs the first number it sees). In AI mode these are handled by understanding the sentence.

In [ ]:
sentences = [
    "Spent 12.50 on lunch with friends yesterday",
    "Bus ticket 3",
    "Paid 60 for the electricity bill",
    "Bought a new t-shirt for 25",
    "Movie night 15 dollars",
    "Pharmacy 18.75 for cold medicine",
    "Weekly groceries 54.20 two days ago",
    "Bought 2 coffees for 9 total",
]

records = []
for s in sentences:
    try:
        rec = parse_expense(s)
        records.append(rec)
        print(f"OK  {s!r}\n    -> {rec['category']}, {rec['amount']}, {rec['date']}")
    except Exception as err:
        print(f"ERR {s!r} -> {err}")

> **Check the output yourself.** In offline mode, "Bought 2 coffees for 9 total" is recorded as **2.0**, not 9. That's the kind of mistake you must look for. AI makes different mistakes, so verify its results too.

### Step 5: Analyze with Pandas

In [ ]:
df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"])
df

In [ ]:
print(f"Total spent:   ${df['amount'].sum():,.2f}")
print(f"Average spend:  ${df['amount'].mean():,.2f}")
print(f"Biggest expense: {df.loc[df['amount'].idxmax(), 'description']} (${df['amount'].max():,.2f})")

by_category = df.groupby("category")["amount"].sum().sort_values(ascending=False)
by_category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].bar(by_category.index, by_category.values, color="steelblue")
axes[0].bar_label(bars, labels=[f"${v:,.2f}" for v in by_category.values], padding=3, fontsize=9)
axes[0].set_title("Spending by Category")
axes[0].set_ylabel("Amount ($)")
axes[0].tick_params(axis="x", rotation=30)

# Pie chart
axes[1].pie(by_category.values, labels=by_category.index, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white"})
axes[1].set_title("Share of Spending")

plt.tight_layout()
plt.show()

### Step 6: AI-generated insights
We give the AI **our computed numbers** (not raw guesses) and ask for advice. Because the facts come from our own code, the AI can't invent the data. It only explains it.

In [ ]:
def get_insights(df):
    summary = df.groupby("category")["amount"].agg(["count", "sum"]).round(2).to_string()
    total = df["amount"].sum()

    if USE_AI:
        prompt = f"""You are a friendly budgeting coach for a student.
Here is their spending summary (total ${total:.2f}):

{summary}

Give exactly 3 short, practical tips based ONLY on these numbers.
Use a bullet list, no more than 25 words per tip."""
        try:
            return ask_ai(prompt)
        except Exception as err:
            print(f"(AI failed: {err}. Using simple summary.)")

    # Offline fallback: simple rules
    top = df.groupby("category")["amount"].sum().idxmax()
    share = df.groupby("category")["amount"].sum().max() / total
    return (f"- Your biggest category is {top} ({share:.0%} of spending).\n"
            f"- Average expense is ${df['amount'].mean():.2f}; look for small ones that add up.\n"
            f"- Set a monthly limit for {top} and track it weekly.")

print(get_insights(df))

### Step 7: Try your own sentence
Run this cell, type an expense in plain English, and see how it's parsed. Run it again to add more.

In [ ]:
text = input("Describe an expense (e.g. 'taxi to airport 32 yesterday'): ")
try:
    rec = parse_expense(text)
    print("Parsed:", rec)
    df = pd.concat([df, pd.DataFrame([rec])], ignore_index=True)
    print(f"Total expenses: {len(df)} | Total spent: ${df['amount'].sum():,.2f}")
except Exception as err:
    print("Couldn't understand that:", err)

---
# Reflection & Challenges

## What you practiced
- Writing a **structured prompt** (role, task, context, format)
- Getting **JSON** out of an AI so a program can use it
- **Validating** AI output instead of trusting it
- Adding a **fallback** so the app works when the AI fails
- Giving the AI **real computed data** to reduce hallucination

## Challenges
1. Add a **budget limit** per category and have the AI warn you when you're close.
2. Let one sentence contain **multiple expenses** ("coffee 4 and a bus ticket 3") and return a JSON *list*.
3. Save the data to **CSV** (`df.to_csv(...)`) and reload it next session.
4. Add a **chat cell** where you ask questions like "How much did I spend on food?" Have the AI answer using your DataFrame summary.
5. Swap Gemini for another provider's API and compare accuracy on the same sentences.

## AI usage log (fill this in)
| Task | Tool used | Prompt I wrote | Was the answer correct? | What I changed |
|---|---|---|---|---|
| | | | | |
| | | | | |